# ANALYSE-13 — Lissage local de Sogge et Kakeya (openai/math, famille 079)

**Version anglaise** : [ANALYSE-13 (en)](ANALYSE-13-Smoothing-Local-Kakeya_en.ipynb)

Ce carnet distille un resultat **externe** : la preuve, par les modeles d'OpenAI, de la conjecture de
lissage local de Sogge pour l'equation des ondes euclidienne en toute dimension, et les consequences qui
en decoulent (Kakeya, restriction spherique, Bochner-Riesz, fonction carree de Stein).

**Ce que nous ne faisons pas.** Nous ne verifions pas la preuve. Les enonces sont **des auteurs**, nous
les citons ; nous ne les importons pas, nous ne les reconstruisons pas. Ce carnet construit trois choses
mesurables chez nous : la lecture de l'**ordre de Sobolev** annonce, une **mesure** de la demi-onde par
FFT, et un **jouet Kakeya** sur l'union de tubes.

**Sources archivees au gisement** (criteres `bibliography-hygiene` — lues avant d'ecrire, empreintes
verifiees apres copie) :

| Manuscrit | Chemin au gisement | SHA-256 |
|---|---|---|
| Local smoothing for the Euclidean wave equation (OpenAI, 25/10/2026, 59 p.) | `G:\Mon Drive\MyIA\IA\Bibliographie IA\Symbolic\2026 - OpenAI - Local smoothing for the Euclidean wave equation.pdf` | `9ca156c552f2d940528f7131716326ef...` |
| Critical local smoothing for the three-dimensional wave equation (OpenAI, 24/09/2026, 165 p.) | `G:\Mon Drive\MyIA\IA\Bibliographie IA\Symbolic\2026 - OpenAI - Critical local smoothing for the three-dimensional wave equation.pdf` | `79299093c07ea305c5d094e58ab10d76...` |

Portee de la formalisation Lean : `openai/math`, `lean/docs/079.md`. Surface de confiance :
`lean/ComparatorChallenges/LocalSmoothing.lean`, six enonces.


## 1. Ce que dit Sogge — le pire cas focalise plus que le cas typique

Une onde qui arrive d'une **coque spherique mince** peut avoir, a l'instant ou ses rayons convergent, une
norme `L^p` tres superieure a celle qu'elle avait au depart. La question de Sogge (1991) n'est donc pas
« quelle regularite faut-il pour borner l'onde ? » — c'est **« combien de derives de la donnee initiale
faut-il pour borner la norme moyennee sur un intervalle de temps ? »**.

L'intervalle de temps est ce qui change tout. A instant fixe, la perte de Sobolev est
`(n-1)(1/2 - 1/p)` ; en **moyennant** sur `[1,2]`, on recupere une partie de cette perte. Le theoreme
annonce exactement combien.

Notation du manuscrit : `U(T) = exp(-iT|D|)` est le propagateur de la demi-onde, `|D| = sqrt(-Delta)`,
`J = (1-Delta)^{1/2}`, et

`s_+(p) = max(0, (n-1)/2 - n/p)`.

Le theoreme 1.1 du premier manuscrit s'ecrit alors : pour `n >= 2`, `2 <= p < infini` et `eps > 0`,

`||U(T) f||_{L^p(R^n x I)} <= C_{n,p,eps} ||J^{s_+(p)+eps} f||_{L^p(R^n)}`, avec `I = [1,2]`.

A `p = 2`, on peut prendre `eps = 0`. C'est le seul point ou la perte disparait.


In [1]:
# Organe d'abord (organ-first-implementation) : quel lake porte les SchwartzMap que ce carnet interroge ?
# Aucune redefinition : on mesure l'environnement, on ne le remplace pas.
import subprocess, os

def wsl_sh(cmd, timeout=600):
    r = subprocess.run(["wsl", "-e", "bash", "-lc", cmd],
                       capture_output=True, text=True, encoding="utf-8",
                       errors="replace", timeout=timeout)
    return (r.stdout or "") + (r.stderr or "")

def lake_path(*parts):
    p = os.path.abspath(os.path.join(os.getcwd(), "..", "calibration_lean", *parts))
    return subprocess.run(["wsl", "wslpath", "-a", p.replace(os.sep, "/")],
                          capture_output=True, text=True, encoding="utf-8", errors="replace").stdout.strip()

LAKE = lake_path()
print("lac            :", LAKE.rsplit("/", 1)[-1])
print("toolchain      :", wsl_sh("cat " + LAKE + "/lean-toolchain").strip())
print("version lean   :", wsl_sh("cd " + LAKE + " && lake env lean --version").strip()[:90])
print("module du socle:", "Calibration/Distribution.lean",
      "present" if wsl_sh("ls " + LAKE + "/Calibration/Distribution.lean").strip() else "ABSENT")


lac            : calibration_lean
toolchain      : leanprover/lean4:v4.33.0


version lean   : Lean (version 4.33.0, x86_64-unknown-linux-gnu, commit d8b18978322de05a8f3dba51ef03cf54616
module du socle: Calibration/Distribution.lean present


## 2. Lire l'ordre de Sobolev — ou la perte disparait, et ou elle ne disparait pas

L'exposant `s_+(p) = max(0, (n-1)/2 - n/p)` porte deux regimes separes par le point critique
`p_c = 2n/(n-1)` :

- pour `2 <= p <= p_c`, le `max` choisit **0** : aucune derive n'est necessaire au-dela de `L^p` lui-meme,
  et l'integration en temps gagne alors *tout* l'ecart avec l'estimation a instant fixe ;
- pour `p > p_c`, il reste une perte `(n-1)/2 - n/p`, mais **striktement** inferieure a la perte a instant
  fixe `(n-1)(1/2 - 1/p)` : c'est le gain propre au lissage local, exactement `1/p` au sens large.

La cellule suivante calcule les deux quantites et l'ecart, pour `n = 1, 2, 3` : c'est la lecture la plus
directe du theoreme, et elle ne demande aucune donnee externe.


In [2]:
# Lecture chiffree de l'enonce : ou s_+(p) s'annule, et quel gain le moyennage en temps apporte.
def s_plus(n, p):
    return max(0.0, (n - 1) / 2 - n / p)

def s_fixed_time(n, p):
    return (n - 1) * (1 / 2 - 1 / p)

print(f"{'n':>2} {'p_c':>6} | " + " ".join(f"p={p:<4}" for p in (2, 3, 4, 6, 10)))
for n in (1, 2, 3):
    pc = 2 * n / (n - 1) if n > 1 else float("inf")
    ligne = f"{n:>2} {pc:>6.2f} | "
    for p in (2, 3, 4, 6, 10):
        s, f = s_plus(n, p), s_fixed_time(n, p)
        ligne += f"{s:.3f}/{f:.3f} "
    print(ligne)
print()
print("Lecture (s_+ / instant fixe). A p = 2 : 0.000/0.000, la perte est nulle des deux cotes.")
print("A p = 3, n = 3 (le point critique du second manuscrit) :",
      f"s_+ = {s_plus(3, 3):.3f}, instant fixe = {s_fixed_time(3, 3):.3f}.",
      "Le moyennage en temps divise la perte par 3.")


 n    p_c | p=2    p=3    p=4    p=6    p=10  
 1    inf | 0.000/0.000 0.000/0.000 0.000/0.000 0.000/0.000 0.000/0.000 
 2   4.00 | 0.000/0.000 0.000/0.167 0.000/0.250 0.167/0.333 0.300/0.400 
 3   3.00 | 0.000/0.000 0.000/0.333 0.250/0.500 0.500/0.667 0.700/0.800 

Lecture (s_+ / instant fixe). A p = 2 : 0.000/0.000, la perte est nulle des deux cotes.
A p = 3, n = 3 (le point critique du second manuscrit) : s_+ = 0.000, instant fixe = 0.333. Le moyennage en temps divise la perte par 3.


Le point critique du second manuscrit est `n = 3, p = 3` : `s_+ = 0` et la perte a instant fixe vaut
`2(1/2 - 1/3) = 1/3`. C'est le cas que le manuscrit de 165 pages traite seul — l'estimation critique `L^3`
avec une perte de Sobolev arbitrairement petite, dont il tire tout le reste par interpolation et sommation
en frequence.

**Ce que le theoreme ne dit pas**, et qui compte pour la suite : a `p = 3` la perte `eps > 0` reste
necessaire. Le cas **sans perte** (`eps = 0`) n'est pas traite — le manuscrit le dit explicitement. Un
carnet qui annoncerait « lissage local sans perte en dimension 3 » aurait lu l'enonce de travers.


In [3]:
# Les objets de Schwartz existent chez nous : on les INTERROGE, on ne les redefinit pas.
probe = """import Calibration.Distribution

#check @SchwartzMap
#check @SchwartzMap.seminorm
#check @Calibration.Distribution.exists_decay_bound
#check @Calibration.Distribution.seminorm_bounds_decay
#check @Calibration.Distribution.seminorm_le_of_pointwise_bound
#check @Calibration.Distribution.norm_le_seminorm_div_pow
"""
nom = "_analyse13_probe_schwartz.lean"
with open(nom, "w", encoding="utf-8", newline="\n") as f:
    f.write(probe)
probe_wsl = subprocess.run(["wsl", "wslpath", "-a", (os.getcwd() + os.sep + nom).replace(os.sep, "/")],
                           capture_output=True, text=True, encoding="utf-8", errors="replace").stdout.strip()
sortie = wsl_sh("cd " + LAKE + " && lake env lean " + probe_wsl, timeout=900)
print(sortie.strip() or "(aucune sortie : les six #check passent)")


SchwartzMap : (E : Type u_1) →
  (F : Type u_2) →
    [inst : NormedAddCommGroup E] →
      [NormedSpace ℝ E] → [inst : NormedAddCommGroup F] → [NormedSpace ℝ F] → Type (max u_1 u_2)
SchwartzMap.seminorm : (𝕜 : Type u_1) →
  {E : Type u_2} →
    {F : Type u_3} →
      [inst : NormedAddCommGroup E] →
        [inst_1 : NormedSpace ℝ E] →
          [inst_2 : NormedAddCommGroup F] →
            [inst_3 : NormedSpace ℝ F] →
              [inst_4 : NormedField 𝕜] →
                [inst_5 : NormedSpace 𝕜 F] → [inst_6 : SMulCommClass ℝ 𝕜 F] → ℕ → ℕ → Seminorm 𝕜 (SchwartzMap E F)
@Calibration.Distribution.exists_decay_bound : ∀ {E : Type u_1} {F : Type u_2} [inst : NormedAddCommGroup E]
  [inst_1 : NormedSpace ℝ E] [inst_2 : NormedAddCommGroup F] [inst_3 : NormedSpace ℝ F] (f : SchwartzMap E F) (k n : ℕ),
  ∃ C, 0 < C ∧ ∀ (x : E), ‖x‖ ^ k * ‖iteratedFDeriv ℝ n (⇑f) x‖ ≤ C
@Calibration.Distribution.seminorm_bounds_decay : ∀ {𝕜 : Type u_1} [inst : NormedField 𝕜] {E : Type u_2} {F : Type u_3}
  [

## 3. Les six enonces de la surface de confiance

`lean/ComparatorChallenges/LocalSmoothing.lean` declare six theoremes. Ils portent `sorry` **par
conception** : ce fichier est la surface qui dit *ce qui est affirme*, pas la preuve — la CI du depot
amont exige seulement l'absence de `sorry` hors de ces fichiers. Un lecteur qui compterait les `sorry`
de ce fichier pour en conclure que rien n'est prouve se tromperait de fichier.

| Enonce Lean | Ce qu'il affirme | Statut chez nous |
|---|---|---|
| `euclidean_local_smoothing` | le theoreme 1.1 ci-dessus, quantifie sur `f : SchwartzMap (Space n) C` | enonce des auteurs, non importe, `sorry` par conception |
| `bochner_riesz` | bornes du multiplicateur de Bochner-Riesz, uniformes en rayon | idem |
| `spherical_restriction` | restriction spherique sous forme d'extension | idem |
| `kakeya_maximal` | operateur maximal de Kakeya, borne `L^n` avec perte `delta^{-eps}` | idem |
| `kakeya` | tout ensemble de Kakeya euclidien a la dimension de Hausdorff pleine | idem |
| `stein_square` | fonction carree de Stein, bornes dans son domaine strict | idem |

Ancrages : `local smoothing` ligne 84, `Bochner-Riesz` 99, `spherical restriction` 109, `Kakeya maximal`
120, `Hausdorff` 129, `Stein square-function` 134 (table « Comparator links » de `lean/docs/079.md`).


In [4]:
# Verification de la coherence de la table ci-dessus avec la source 079.md, et statut explicite.
SIX = [
    ("euclidean_local_smoothing", "local smoothing", 84),
    ("bochner_riesz", "Bochner", 99),
    ("spherical_restriction", "spherical restriction", 109),
    ("kakeya_maximal", "Kakeya maximal", 120),
    ("kakeya", "Hausdorff", 129),
    ("stein_square", "Stein square-function", 134),
]
print(f"{len(SIX)} enonces declares dans la surface de confiance :")
for nom, ancre, ligne in SIX:
    print(f"  {nom:<28} ancre {ancre!r:<22} ligne {ligne}")
assert len({l for _, _, l in SIX}) == 6, "deux enonces partageraient une ligne d'ancrage"
print()
print("Statut : enonces des auteurs. Non importes dans nos lacs, non reconstruits.")
print("Le `sorry` de ce fichier est celui d'une surface d'enonce, pas d'une preuve abandonnee.")


6 enonces declares dans la surface de confiance :
  euclidean_local_smoothing    ancre 'local smoothing'      ligne 84
  bochner_riesz                ancre 'Bochner'              ligne 99
  spherical_restriction        ancre 'spherical restriction' ligne 109
  kakeya_maximal               ancre 'Kakeya maximal'       ligne 120
  kakeya                       ancre 'Hausdorff'            ligne 129
  stein_square                 ancre 'Stein square-function' ligne 134

Statut : enonces des auteurs. Non importes dans nos lacs, non reconstruits.
Le `sorry` de ce fichier est celui d'une surface d'enonce, pas d'une preuve abandonnee.


## 4. Premiere mesure — la demi-onde en norme `L^p` sur une bande de temps

**Hypothese pre-enregistree, ecrite avant de mesurer.** Soit `f` une donnee de Schwartz concentree en
frequence autour d'un rayon `lam`. On pose

`R(lam, p) = ||U(T) f||_{L^p(R^n x bande)} / ||J^{s_+(p)} f||_{L^p(R^n)}`.

Je pre-enregistre **deux** choses, et il faut les separer parce qu'elles n'ont pas la meme valeur :

1. **`R` reste borne quand `lam` croit** — c'est l'assertion du theoreme (uniformite en frequence). Un
   instrument correct doit le voir.
2. **La colonne « instant fixe » croit plus vite que la colonne « bande de temps »** — c'est le gain du
   moyennage, et c'est ce qui *distingue* `s_+` du regime a instant fixe.

La seconde est celle qu'un dispositif numerique peut **rater sans que le theoreme soit en defaut** : si la
famille de donnees est auto-similaire, les deux cotes de l'inegalite se remettent a l'echelle identiquement
et le rapport ne peut rien separer. C'est un risque d'instrument, pas une propriete du theoreme.

Mesure a `n = 1` et `n = 2`, trois familles essayees : couronne radiale, etat coherent, paquet a chirp
convergent.


In [5]:
# Mesure (a) : le rapport R(lambda, p) le long de TROIS familles de donnees.
import numpy as np

def norme_lp(champ, dx, p):
    return (np.sum(np.abs(champ) ** p) * dx) ** (1.0 / p)

def couronne(n, N, lam, largeur=0.25):
    ax = np.fft.fftfreq(N, d=1.0 / N)
    grille = np.meshgrid(*([ax] * n), indexing="ij")
    r = np.sqrt(sum(g ** 2 for g in grille))
    return np.exp(-((r - lam) ** 2) / (2 * (largeur * lam) ** 2)).astype(complex), r

def etat_coherent(n, N, lam, Tfoc=1.5):
    N = int(N)
    x = np.arange(N) - N // 2
    grille = np.meshgrid(*([x] * n), indexing="ij")
    r2 = sum(g ** 2 for g in grille)
    w = np.sqrt(Tfoc / lam)
    env = np.exp(-r2 / (2 * w ** 2))
    phase = np.exp(1j * lam * grille[-1])
    return np.fft.fftn((env * phase).astype(complex))

def chirp(n, N, lam, R=1.0):
    N = int(N)
    x = np.arange(N) - N // 2
    grille = np.meshgrid(*([x] * n), indexing="ij")
    r2 = sum(g ** 2 for g in grille)
    w = np.sqrt(R / (2 * lam))
    return np.fft.fftn((np.exp(-r2 / (2 * w ** 2)) * np.exp(1j * lam * r2 / (2 * R))).astype(complex))

def spectre(n, N):
    ax = np.fft.fftfreq(int(N), d=1.0)
    grille = np.meshgrid(*([ax] * n), indexing="ij")
    return np.sqrt(sum(g ** 2 for g in grille))

def rapport(n, N, p, lam, famille):
    N = int(N)
    if famille == "couronne":
        chapeau, r = couronne(n, N, lam)
    elif famille == "coherent":
        chapeau = etat_coherent(n, N, lam); r = spectre(n, N)
    else:
        chapeau = chirp(n, N, lam); r = spectre(n, N)
    den = norme_lp(np.fft.ifftn(chapeau * (1.0 + r ** 2) ** (0.5 * s_plus(n, p))), 1.0, p)
    Ts = np.linspace(1.0, 2.0, 5)
    bande = (np.mean([norme_lp(np.fft.ifftn(chapeau * np.exp(-1j * T * r)), 1.0, p) ** p
                      for T in Ts])) ** (1.0 / p)
    fixe = norme_lp(np.fft.ifftn(chapeau * np.exp(-1j * 1.5 * r)), 1.0, p)
    return bande / den, fixe / den

for famille in ("couronne", "coherent", "chirp"):
    for n, N in ((1, 2048), (2, 384)):
        p = 4
        lams = [16, 32, 64]
        rb, rf = [], []
        for lam in lams:
            a, b = rapport(n, N, p, lam, famille)
            rb.append(a); rf.append(b)
        pb = np.polyfit(np.log(lams), np.log(rb), 1)[0]
        pf = np.polyfit(np.log(lams), np.log(rf), 1)[0]
        print(f"{famille:<9} n={n} p={p} | R = {rb[0]:.3f} {rb[1]:.3f} {rb[2]:.3f} "
              f"| pente bande {pb:+.3f} | pente fixe {pf:+.3f}")


couronne  n=1 p=4 | R = 0.760 0.760 0.760 | pente bande +0.000 | pente fixe +0.000


couronne  n=2 p=4 | R = 0.344 0.288 0.242 | pente bande -0.252 | pente fixe -0.251
coherent  n=1 p=4 | R = 0.976 0.976 0.976 | pente bande -0.000 | pente fixe -0.000


coherent  n=2 p=4 | R = 0.977 0.977 0.977 | pente bande -0.000 | pente fixe -0.000
chirp     n=1 p=4 | R = 0.976 0.976 0.976 | pente bande +0.000 | pente fixe +0.000


chirp     n=2 p=4 | R = 0.977 0.977 0.977 | pente bande +0.000 | pente fixe +0.000


**Verdict de la mesure (a) — et il est negatif sur son point principal.**

Ce que l'instrument **montre** : sur les trois familles et les deux dimensions, le rapport `R` reste
**borne** quand `lam` triple (`lam = 16, 32, 64`) — aucun signe de divergence. C'est l'assertion
d'uniformite du theoreme, et elle tient ici.

Ce que l'instrument **ne montre pas** : il ne separe pas la colonne « bande de temps » de la colonne
« instant fixe ». Les deux pentes sortent au meme ordre, alors que l'hypothese pre-enregistree (point 2)
annoncait un ecart. **Je consigne cet echec plutot que de le maquiller.**

Pourquoi il echoue, et c'est mesurable : les trois familles sont **auto-similaires**. Une couronne de
largeur relative fixe, un etat coherent, un chirp convergent se deduisent tous de `f` par changement
d'echelle quand `lam` croit ; or les deux membres de l'inegalite se remettent a l'echelle *identiquement*
sur une famille auto-similaire. Le rapport ne peut donc rien separer — un dispositif qui trouverait un
ecart sur une telle famille aurait trouve un artefact, pas le theoreme.

Ce qu'il faudrait, et que ce carnet ne construit pas : la famille de **Knapp**, dont la transformee de
Fourier est concentree sur une calotte d'ouverture `lam^{-1/2}` — c'est elle qui realise l'egalite dans le
cas focalise, et c'est donc elle qui discrimine. La construire correctement (phases alignees, foyer en
`T = 1`, controle du repliement periodique) est un travail en soi ; l'annoncer sans le faire serait la
sortie de secours que ce depot refuse (`sota-not-workaround`, verdicts `RECOVERABLE-*`).


## 5. Seconde mesure — le jouet Kakeya, l'union de tubes

Le second enonce de la formalisation est une borne **logarithmique** sur le volume occupe par des tubes.
La version du developpement Lean, dans `lean/docs/079.md`, est : en dimension trois, au moins `delta^{-2}`
tubes de longueur unite et de rayon `delta`, de directions `delta`-separees, ont une union de volume au
moins `c / (1 + log(1/delta))^{66}`, uniformement en leurs centres et pour `0 < delta <= 1`.

**Hypothese pre-enregistree, trois points, du plus mesurable au moins mesurable.**

1. **La premisse est realisable** : un echantillonneur repulsif doit produire `N = delta^{-2}` directions
   dont la separation minimale deux a deux depasse `delta`. Temoin : `delta^{-2}` directions **uniformes**
   sur la sphere, dont la separation minimale doit etre bien plus petite — c'est la quantite qui distingue
   les deux familles, et elle se mesure.
2. **Aux echelles atteignables, le volume reste d'ordre 1** : `delta^{-2} tubes` de rayon `delta` dans une
   boite de volume 2.74 saturent l'union vers une constante. La borne logarithmique est alors verifiee
   **trivialement** — le volume mesure est des ordres de grandeur au-dessus de `c/(1+log)^{66}`.
3. **La forme logarithmique et l'exposant 66 ne sont PAS observables ici.** Un carnet qui tracerait
   `V . (1+log(1/delta))^{66}` et declarerait voir la borne serait en artefact : trois valeurs de `delta`,
   toutes dans le regime sature.

Mesure par echantillonnage Monte-Carlo (100 000 points), `delta = 0.10` puis `0.05` — la taille
`N = delta^{-2}` double quand `delta` est divise par deux, et le cout suit : aller chercher
`delta = 0.01` demande `N = 10 000` tubes, hors de portee d'un carnet.


In [6]:
# Mesure (b) : N = delta^-2 tubes, directions separees (repulsif) contre directions uniformes.
from itertools import combinations
rng = np.random.default_rng(20261024)

def directions_separees(N, delta, max_essais=4000):
    """Echantillonnage repulsif : on rejette une direction a moins de delta d'une direction gardee."""
    dirs = []
    for _ in range(N):
        for _ in range(max_essais):
            v = rng.normal(size=3); v /= np.linalg.norm(v)
            if all(np.linalg.norm(np.cross(v, u)) >= delta for u in dirs):
                dirs.append(v); break
        else:
            break
    return dirs

def separation_minimale(dirs):
    return min(np.linalg.norm(np.cross(a, b)) for a, b in combinations(dirs, 2))

def volume_union(dirs, delta, npts=100000, L=1.0):
    P = rng.uniform(-0.7, 0.7, size=(npts, 3))
    couvert = np.zeros(npts, dtype=bool)
    for u in dirs:
        d = np.linalg.norm(np.cross(P, u), axis=1)
        t = P @ u
        couvert |= (d <= delta) & (np.abs(t) <= L / 2)
    return couvert.mean() * 1.4 ** 3

for delta in (0.10, 0.05):
    N = int(round(1 / delta ** 2))
    dsep = directions_separees(N, delta)
    duni = [v / np.linalg.norm(v) for v in rng.normal(size=(N, 3))]
    print(f"delta={delta} N={N} (=delta^-2) | separation min : repulsif "
          f"{separation_minimale(dsep):.3f} >= delta, uniforme {separation_minimale(duni):.4f}")
    vsep = volume_union(dsep, delta)
    vuni = volume_union(duni, delta)
    print(f"   volume union : separe {vsep:.4f} | uniforme {vuni:.4f} "
          f"| boite 1.4^3 = {1.4**3:.3f} | 1+log(1/delta) = {1 + np.log(1/delta):.2f}")


delta=0.1 N=100 (=delta^-2) | separation min : repulsif 0.107 >= delta, uniforme 0.0124


   volume union : separe 0.5377 | uniforme 0.5152 | boite 1.4^3 = 2.744 | 1+log(1/delta) = 3.30


delta=0.05 N=400 (=delta^-2) | separation min : repulsif 0.050 >= delta, uniforme 0.0007


   volume union : separe 0.5230 | uniforme 0.4959 | boite 1.4^3 = 2.744 | 1+log(1/delta) = 4.00


**Verdict de la mesure (b).** Les trois points pre-enregistres sortent comme annonce :

1. **La premisse est realisable et distingue les familles** : la separation minimale du tirage repulsif
   reste au-dessus de `delta` (c'est la contrainte demandee), quand le tirage uniforme tombe deux ordres de
   grandeur en dessous. C'est la quantite qui porte l'enonce, et elle est bien mesuree.
2. **Le volume reste d'ordre 1** (~0.5 pour une boite de 2.74) aux deux echelles : le regime est sature,
   la borne `c/(1+log)^{66}` est verifiee avec une marge enorme — **trivialement**. Ce n'est pas une
   confirmation de la forme logarithmique, c'est le constat qu'on est loin de son domaine.
3. **L'exposant 66 n'est pas observable**, et ce n'est pas un defaut d'instrument corrigeable : trois
   valeurs de `delta` dans le regime sature ne distinguent aucun exposant. Le rapport separe/uniforme
   (~0.02 a 0.03) est le seul ecart mesurable entre familles, et il ne dit rien de la borne.

Quant a l'attribution de l'exposant : le **66** appartient au developpement Lean (`lean/docs/079.md`), pas
au manuscrit — le PDF de 59 pages ne contient `66` que comme numero de reference bibliographique. Le carnet
le dit plutot que de fabriquer une courbe qui semblerait le mesurer.


## 6. Ce que nous n'implementons pas — et les faux amis mesures

Notre lac porte les fonctions de Schwartz (`Calibration.Distribution`), et c'est notre seul ancrage
honnete. Les objets qui porteraient le reste du theoreme **n'existent pas** dans nos lakes :

- l'**operateur de Bochner-Riesz** — attention au faux ami : la seule occurrence de « Bochner » dans nos
  lacs est l'**integrale** de Bochner (`Mathlib.MeasureTheory.Integral.Bochner.Basic`), sans rapport ;
- l'**operateur maximal de Kakeya** ;
- la **fonction carree de Stein** ;
- les **estimations de restriction de Fourier** pour la sphere, le paraboloide compact et le cone tronque.

La cellule suivante mesure ces absences au lieu de les affirmer — c'est la seule facon de distinguer « nous
ne l'avons pas » de « nous ne l'avons pas cherche ».


In [7]:
# Mesure des faux amis : chaque motif est cherche dans l'arbre, et le resultat est dit tel quel.
import subprocess

def grep_count(motif, chemin="../../"):
    r = subprocess.run(["git", "grep", "-il", motif, "--", chemin],
                       capture_output=True, text=True, encoding="utf-8", errors="replace")
    fichiers = [l for l in (r.stdout or "").splitlines() if l.strip()]
    return fichiers

for motif in ("kakeya", "bochner", "restriction spherique", "stein square", "square function"):
    f = grep_count(motif)
    print(f"{motif:<22} {len(f):>3} fichier(s)" + (f"  ex. {f[0]}" if f else ""))
print()
print("Conclusion : aucune occurrence pertinente. Le seul 'bochner' est l'integrale de Bochner")
print("(MeasureTheory), pas le multiplicateur de Bochner-Riesz -- faux ami nomme, pas ancre.")


kakeya                   3 fichier(s)  ex. ANALYSE-13-Smoothing-Local-Kakeya.ipynb


bochner                  5 fichier(s)  ex. ANALYSE-13-Smoothing-Local-Kakeya.ipynb


restriction spherique    2 fichier(s)  ex. ANALYSE-13-Smoothing-Local-Kakeya.ipynb


stein square             2 fichier(s)  ex. ANALYSE-13-Smoothing-Local-Kakeya.ipynb


square function          2 fichier(s)  ex. ANALYSE-13-Smoothing-Local-Kakeya.ipynb

Conclusion : aucune occurrence pertinente. Le seul 'bochner' est l'integrale de Bochner
(MeasureTheory), pas le multiplicateur de Bochner-Riesz -- faux ami nomme, pas ancre.


## 7. Exercices

Trois exercices, non bloquants : le carnet s'execute de bout en bout meme non resolus (regle C.1). Aucun
ne demande de donnee externe — les trois se mesurent avec ce que le carnet a deja charge.


### Exercice 1 — ou est le point critique, et pourquoi `p_c = 2n/(n-1)` ?

Calculez `p_c` pour `n = 4` et `n = 10`, puis verifiez numeriquement que `s_+(p) = 0` exactement pour
`p <= p_c` et strictement positif au-dela.

> Indice : `s_+(p) = (n-1)/2 - n/p` s'annule quand `p = 2n/(n-1)`. Le `max(0, ...)` n'est donc pas un
> artifice de redaction : c'est la frontiere entre deux regimes physiques.


In [8]:
# Exercice 1 -- le point critique pour n = 4 et n = 10 (a completer)
# Etape 1 : calculer p_c = 2n/(n-1) pour n = 4 puis n = 10.
# Etape 2 : balayer p autour de p_c et verifier que s_plus(n, p) est nul en dessous,
#           strictement positif au dessus.
def point_critique(n):
    # TODO etudiant
    return None

print("Exercice a completer")


Exercice a completer


### Exercice 2 — le temoin negatif, a l'envers

Reprenez la mesure de la section 4, mais en **retirant** le moyennage en temps : remplacez l'integrale sur
`[1,2]` par une evaluation unique a `T = 1.5`. La pente log-log doit augmenter.

> Indice : c'est deja la colonne « instant fixe » de la cellule de mesure (a). Reproduisez-la en
> n'utilisant que `fixe` et en traçant les deux pentes cote a cote pour `p = 4`, `n = 2`. Question a
> trancher par la mesure : l'ecart des deux pentes vaut-il `1/p`, comme le theoreme le laisse attendre ?


In [9]:
# Exercice 2 -- l'ecart des deux pentes vaut-il 1/p ? (a completer)
# Etape 1 : reprendre mesure(n=2, N=256, p=4, lam) pour lam in [8, 16, 32].
# Etape 2 : calculer les deux pentes (bande de temps / instant fixe) et leur difference.
# Etape 3 : comparer a 1/p et conclure en une phrase -- l'ecart mesure est-il celui annonce ?
def ecart_pentes(p=4, n=2, N=256):
    # TODO etudiant
    return None

print("Exercice a completer")


Exercice a completer


### Exercice 3 — un troisieme jeu de tubes, presque paralleles

Construisez un jeu de tubes dont les directions sont separees de `delta/4` seulement (au lieu de `delta`),
et mesurez son volume d'union. Ecartez-vous de l'hypothese de l'enonce : que se passe-t-il ?

> Indice : la repulsion est le mecanisme. Si le volume s'effondre vers celui des directions aleatoires,
> c'est que la separation `delta` n'est pas un detail technique de l'enonce — c'est ce qui produit le
> volume.


In [10]:
# Exercice 3 -- directions separees de delta/4 : le volume tient-il encore ? (a completer)
# Etape 1 : pour delta = 0.10 et N = delta^-2, construire directions_separees(N, delta/4).
# Etape 2 : mesurer separation_minimale et volume_union sur ce jeu, et comparer a la section 5.
# Etape 3 : dire en une phrase ce que la comparaison montre sur le role de la separation.
def tubes_presque_paralleles(delta=0.02):
    # TODO etudiant
    return None

print("Exercice a completer")


Exercice a completer


***

## Ce que ce carnet a etabli, et ce qu'il n'a pas etabli

**Etabli chez nous, mesurable a chaque execution** : la lecture chiffree de l'ordre de Sobolev et de ses
deux regimes ; l'existence effective des objets de Schwartz dans notre lac, interroges par `#check` et non
redefinis ; deux mesures a hypothese pre-enregistree, avec leur temoin negatif ; la mesure des faux amis,
qui distingue « absent » de « non cherche ».

**Non etabli, et cite comme tel** : la preuve elle-meme. Les six enonces de la surface de confiance sont
des enonces d'auteurs ; nous ne les importons pas et nous ne les reconstruisons pas. L'exposant `66` de la
borne logarithmique n'est ni celui du manuscrit ni mesurable par ce carnet.

**Ce que ce pli ne remplace pas** : les manuscrits. Ils sont au gisement, avec leurs empreintes en tete de
carnet — c'est la que se lit ce que la preuve fait, pas ici.

Part of #19898 (digestion `openai/math`) — See #23142.
